In [ ]:
from pathlib import Path
import xarray as xr
import fsspec

import zarr
from zarr.codecs import BloscCodec

from dask.distributed import Client

xr.set_options(use_new_combine_kwarg_defaults=True);

In [ ]:
ZARR_ARCHIVE = Path("/nvm9/data/swann/zarr_archive/")
water_year = 2026

In [ ]:
CHUNKS = {"time": 30, "lat": 1035, "lon": 1405}

In [ ]:
dask_args = dict(n_workers=12, threads_per_worker=1, dashboard_address="0.0.0.0:8797")
cluster = Client(**dask_args)

In [ ]:
fs = fsspec.filesystem("https")

fsspec_caching = {
    "cache_type": "blockcache",
    "block_size": 10
    * 1024
    * 1024,  # size in bytes per block, recommended are multiple MB
}

In [ ]:
nc_files = [
    fs.open(file, **fsspec_caching)
    for file in fs.glob(
        f"https://climate.arizona.edu/data/UA_SWE/DailyData_800m/WY{water_year}/UA_SWE_Depth_800m_v1_*.nc"
    )
]

In [ ]:
ds = xr.open_mfdataset(
    nc_files,
    preprocess=lambda ds: ds[["SWE", "crs"]],
    engine="h5netcdf",
    chunks=None,  # Will do the chunking later
    parallel=True,
)

In [ ]:
ds = ds.chunk(CHUNKS)
ds

In [ ]:
compression = BloscCodec(cname="lz4", clevel=5, shuffle=zarr.codecs.BloscShuffle.shuffle)
# Reset all info on original ds
for var in ds.variables:
    ds[var].encoding = {}

encoding = {}
for var_name in ds.data_vars:
    if ds[var_name].ndim == 3:
        encoding[var_name] = {
            "chunks": (1, 1035, 1405),
            "shards": (30, 1035, 1405),
            "compressors": compression,
        }
    else:
        encoding[var_name] = {"chunks": ds[var_name].shape}

for coord_name in ds.coords:
    encoding[coord_name] = {"compressors": compression}

In [ ]:
encoding

## Inspect current archive

In [ ]:
if ZARR_ARCHIVE.exists():
    zarr_ds = xr.open_zarr(ZARR_ARCHIVE / "swe.zarr")

In [ ]:
zarr_ds

## Create a new archive

In [ ]:
ds.to_zarr(
    (ZARR_ARCHIVE / f"wy{water_year}_ua_swe.zarr").as_posix(),
    mode="w",
    encoding=encoding,
    consolidated=False,
    zarr_format=3,
)

In [ ]:
cluster.shutdown()

## Delete a variable 

In [ ]:
ds = zarr.open(file.as_posix(), mode="a")
del ds["grid_mapping"]
zarr.consolidate_metadata(file.as_posix())

### Add a CBRFC Zone mask

In [ ]:
ua_mask = xr.open_dataset("/nvm9/data/swann/cbrfc_zone_raster_ua_swe.nc").Band1

In [ ]:
ua_mask.name = "cbrfc_zone_gid"
ua_mask.attrs["long_name"] = "CBRFC zone database gid"

In [ ]:
ua_mask = ua_mask.to_dataset()
ua_mask

In [ ]:
# for archive in ZARR_ARCHIVE.glob("*.zarr"):
ua_mask.to_zarr((ZARR_ARCHIVE / f"wy{water_year}_ua_swe.zarr").as_posix(), mode="a")
zarr.consolidate_metadata((ZARR_ARCHIVE / f"wy{water_year}_ua_swe.zarr").as_posix())

## Add CRS information

In [ ]:
crs = xr.open_dataset("/nvm9/data/swann/SWE_Mask_800m.nc").crs

In [ ]:
crs

In [ ]:
for archive in ZARR_ARCHIVE.glob("*.zarr"):
    crs.to_zarr(archive, mode="a")
    zarr.consolidate_metadata(archive)

In [ ]:
xr.open_mfdataset(ZARR_ARCHIVE.glob("*.zarr"))